In [0]:
IPS = [
    "8.8.8.8",          # Google DNS - US
    "1.1.1.1",          # Cloudflare/APNIC - Australia
    "77.88.8.8",        # Yandex - Russia
    "180.76.76.76",     # Baidu - China
    "168.126.63.1",     # KT - South Korea
    "121.58.203.4",     # PLDT DNS - Philippines
    "84.200.69.80",     # DNS.WATCH - Germany
]

In [0]:
import requests
import time

In [0]:
def get_location_by_ip(ip):
    resp = requests.get(f"http://ip-api.com/json/{ip}", timeout=10)
    resp.raise_for_status()
    data = resp.json()
    if data.get("status") != "success":
        raise ValueError(f"Lookup failed for {ip}: {data.get('message')}")
    return {
        "ip": data["query"],
        "city": data["city"],
        "region": data["regionName"],
        "country": data["country"],
        "lat": data["lat"],
        "long": data["lon"],
        "isp": data["isp"],
    }

In [0]:
locations = []
for ip in IPS:
    try:
        loc = get_location_by_ip(ip)
        locations.append(loc)
        print(f"Added {ip} to locations as {loc["city"]}, {loc["country"]}")
    except Exception as e:
        print(f"Skipping {ip}: {e}")
    time.sleep(1)

locations_df = spark.createDataFrame(locations)
# locations_df.display()

In [0]:
locations_df.write.mode("overwrite").saveAsTable("bronze.locations")